# MeteoScreening `G_FF1_0.05_1` (2021-2025) from database (influxdb)

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `G_FF1_0.05_1` &nbsp;&nbsp;|&nbsp;&nbsp; **Sensor**: Hukseflux HFP01 soil heat flux plate, forest floor FF1, 0.05 m &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2021-2025  
**Derived from**: diive notebook template `DatabaseInfluxStepwiseMeteoScreening.ipynb` (version `10`, 2 Sep 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download raw soil heat flux from the InfluxDB database, screen it on the **high-resolution** data, resample to 30MIN, and upload the result back to the database. Screening uses `StepwiseMeteoScreeningDb` from [diive](https://github.com/holukas/diive). Download and upload use diive's InfluxDB engine (`InfluxIO`).

**Flow:** download (`InfluxIO`) → screen on high-res data (`diive`) → resample to 30MIN → upload.

**Outlier detection is stepwise:** run a test, look at its preview plot, then commit it with `mscr.addflag()`. Re-run with other parameters as often as you like before committing. Run only the tests this variable needs. At the end all committed flags are aggregated into one overall quality flag `QCF`.

The FF1 plot has two HFP01 plates at 0.05 m, `_1` and `_2`. Both log since 26 March 2021. This notebook screens one of them. The other has its own notebook.

Comparisons between the plates, the heat stored in the soil above them, and gap filling are done in `30_PRODUCTS/`, on the 30MIN data.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`**. The stamp marks the end of the averaging interval.

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). It is applied **identically** on download and on upload:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

The timestamps printed and plotted below are therefore local time.

**This matters for `REMOVE_DATES`.** Removal is matched against `TIMESTAMP_MID`, not `TIMESTAMP_END`. A single timestamp such as `'2025-08-15 15:17:00'` therefore matches no record, and nothing is removed and nothing is reported. Write a single minute as the range `['2025-08-15 15:16:00', '2025-08-15 15:17:00']` instead.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night split used during screening.

**Variable to screen**
- `PROFILE`, `DEPTH`, `REPL`: the plate's position tags. `FIELD` is assembled from them and is the InfluxDB `_field`. Change `REPL` to screen the other plate.
- `MEASUREMENT`: exactly **one** measurement grouping the variables. `G` is soil heat flux.

**Time range to screen**
- `START`: first timestamp to screen. It **is** included.
- `STOP`: upper bound. It is **not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the timestamp knob, see *Timestamp convention*. It must match how the raw data was logged (`1` for CET winter time).
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Resampling**
- `RESAMPLING_FREQ`: the screened high-res data is resampled to this frequency.
- `RESAMPLING_AGG`: `'mean'` here. A heat flux is a rate, so it is averaged and never summed.

**Physical range**
- `G_MIN`, `G_MAX`: the limits for the absolute-limits test.

**Parameter help**
- `SHOW_PARAM_HELP`: set `True` to print the full docstring of each screening method right before it runs.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# --- Variable to screen ---
PROFILE = 'FF1'  # forest floor plot
DEPTH = '0.05'  # plate depth in m
REPL = '1'  # <-- the plate, _1 or _2
FIELD = f'G_{PROFILE}_{DEPTH}_{REPL}'
FIELDS = [FIELD]  # StepwiseMeteoScreeningDb expects a list
MEASUREMENT = 'G'

# --- Time range to screen ---
# The plates start 2021-03-26; asking for 2021-01-01 simply starts at the first record.
START = '2021-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Resampling ---
RESAMPLING_FREQ = '30min'  # screened high-res data is resampled to this frequency
RESAMPLING_AGG = 'mean'  # a flux is a rate, never summed

# --- Physical range, in W m-2 ---
# The plates measure roughly -75 to +127 W m-2 over 2021-2025. These limits sit outside
# that, so the test only catches values that cannot be a soil heat flux at all.
G_MIN, G_MAX = -150, 150

# --- Parameter help ---
SHOW_PARAM_HELP = False

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket, e.g. 'ch-lae_processed'
print(f'Screening variable:             {FIELD}')
print(f'Source bucket (raw data):       {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional. Lists all fields available in the measurement, without checking the selected time range. Measurement `G` also holds three older plates, `G_M1_0.05_1` to `G_M3_0.05_1`, which stop at the March 2021 logger rebuild. Those are different sensors and are not screened here:

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its database tags**. This is what the screening reads.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

Drop any requested variable that has no data in this period:

In [ ]:
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} from FIELDS (no data in this period).')
print(f'Data available for: {list(data_detailed.keys())}')

### Verify download timestamps
The timestamps should be in **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`) and mark the **end** of each averaging interval. Compare the first and last stamps against the `START` and `STOP` you asked for.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## ▶️ Start MeteoScreening with `diive`

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Run a test, look at its preview, then commit it with `mscr.addflag()`. Only the committed flag of the **most recent** test is added. Skip any test this variable does not need.

Committed here: **absolute limits** and **missing values**. Manual removal is set up but empty.

The statistical tests are off. This plot is under a deciduous canopy. Before the leaves come out, sunlight reaches the forest floor and the plates read much higher than in summer. Those spring values are real, and a test that compares each value against the whole record would remove them.

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Manual removal
Flag specific timestamps or time ranges for removal, such as known sensor failures or maintenance windows. Give `[start, stop]` pairs.

Empty for now. Write a single minute as a one-minute range, not as a single timestamp. See *Timestamp convention*.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.ManualRemoval)

In [ ]:
REMOVE_DATES = []

if REMOVE_DATES:
    mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)
else:
    print('No manual removal windows - skip the addflag cell below.')

In [ ]:
if REMOVE_DATES:
    mscr.addflag()

### Absolute limits
Flags values outside a fixed physical range `[minval, maxval]`. Here the range only catches values that cannot be a soil heat flux at all, such as a decode error.

Do not clip instead. Clipping replaces a bad value with a plausible one, which nothing downstream can then recognise as false.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.AbsoluteLimits)

In [ ]:
mscr.flag_outliers_abslim_test(minval=G_MIN, maxval=G_MAX, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Other tests (all off)
Off for the reason given at the top of *Outlier detection*. Uncomment one to try it, and look at its preview before committing it.

In [ ]:
# mscr.flag_outliers_hampel_test(window_length=60 * 24, n_sigma=8, use_differencing=True,
#                                separate_day_night=False,
#                                repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_test(thres_zscore=4.5, separate_day_night=False,
#                                repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_rolling_test(thres_zscore=4.5, winsize=60 * 24 * 7,
#                                        repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_localsd_test(n_sd=7, winsize=60 * 24 * 7, constant_sd=False,
#                                 separate_day_night=False,
#                                 repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)
# mscr.addflag()

### Missing values
Not an outlier test. It flags missing records so they are counted in the overall `QCF`.

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Aggregate all committed test flags into one overall flag `QCF` (0 = good, 1 = marginal, 2 = bad) and filter the series. This is required before corrections and resampling.

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 🔧 Corrections
Applied to the high-res, QCF-filtered data. None of them apply to a soil heat flux plate, so every call below stays commented out.

Two of them can look applicable and are not. The nighttime zero-offset correction is for radiation sensors, whose true nighttime value is zero. A buried plate still measures at night. Setting the exact value `0` to missing would delete the two moments each day when the flux changes direction.

In [ ]:
mscr.showplot_cleaned()

Look at the most frequent values first (read-only, safe to run):

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} ({mscr.series_hires_cleaned[ff].count():,} records, '
          f'{mscr.series_hires_cleaned[ff].nunique():,} distinct) ---')
    print(vc.head(20))

In [ ]:
# All commented out on purpose - none of these apply to a soil heat flux plate.
# mscr.correction_remove_nighttime_zero_offset()
# mscr.correction_setto_max_threshold(threshold=150)
# mscr.correction_setto_min_threshold(threshold=-150)
# mscr.correction_setto_value(dates=[['2021-04-01', '2021-04-05']], value=0, verbose=1)
# mscr.correction_set_exact_value_to_missing(values=[0])
# mscr.showplot_cleaned(interactive=False)

## 🔁 Resampling

### Resample to 30MIN
Resample the screened high-res series to `RESAMPLING_FREQ`. The output timestamp is `TIMESTAMP_END` again (see *Timestamp convention*), ready for upload.

In [ ]:
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: {freq}')

## ⬆️ Upload data to database

**Re-uploading overwrites the same variant, so it is safe to re-run.** With `delete_from_db_before_upload=True` (below), the upload first *deletes*, then writes. The delete is scoped to the exact match `_measurement` + `varname` + `data_version` (`meteoscreening_diive`) over the uploaded time range, so re-screening a period replaces only its previous screened result. It never touches the raw data (different `data_version`, and a different `_raw` bucket), other variables, or other data versions. The delete-first step matters because InfluxDB keys a point by its full tag set. If a tag changed between runs (e.g. `units`, `gain`, `offset`), a plain overwrite would leave the old point in place as a **duplicate**. The delete removes it whatever its tags are.

One thing to know for this variable. `ch-lae_processed` already holds `G_FF1_0.05_1` under `meteoscreening_mst`, covering 2004-2011 from an older plate. That is a different data version, so this upload leaves it alone. The two must not be merged by name, because they are different sensors.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the uploaded data again and check its time resolution and timestamps.

In [ ]:
# Fresh variable names so the screened originals (data_detailed etc.) are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes

- Units are W m-2, `gain 1.0`, `offset 0.0`, filegroup `12_meteo_forestfloor`.
- Raw resolution is 1MIN throughout. The plates only ever logged on the CR1000 installed in March 2021, so this record has a single time resolution.
- The record starts 2021-03-26. The first quarter of 2021 is empty because nothing was logged yet.
- Large values in spring are real. Before the leaves come out, sunlight reaches the forest floor.
- This notebook exports gaps where the logger was down and does not fill them.

### ♻️ Reusing this notebook
Copy it and change `REPL` in *User settings* to screen the other plate. `REMOVE_DATES` belongs to one sensor, so do not carry windows across.